In [ ]:
import base64
import mimetypes
from IPython.display import HTML
from google.colab import files

print("Upload driving video")
upload = files.upload()
VIDEO_PATH = next(iter(upload.keys()))

mime_type = mimetypes.guess_type(VIDEO_PATH)[0] or "video/mp4"
with open(VIDEO_PATH, "rb") as f:
    video_data_url = f"data:{mime_type};base64," + base64.b64encode(f.read()).decode("utf-8")

print("video ready:", VIDEO_PATH)


Upload driving video


Saving d0.mp4 to d0.mp4
video ready: d0.mp4


In [ ]:
html = r'''
<div id="mp-raccoon-root" style="font-family: Arial, sans-serif; color: #f5f5f5; background: #111827; padding: 16px; border-radius: 14px;">
  <div style="display: flex; gap: 12px; align-items: center; flex-wrap: wrap; margin-bottom: 12px;">
    <button id="loadBtn" style="padding: 10px 14px; border: 0; border-radius: 10px; background: #d9ff59; color: #111; font-weight: 700; cursor: pointer;">Load Demo</button>
    <button id="playBtn" style="padding: 10px 14px; border: 0; border-radius: 10px; background: #2563eb; color: white; font-weight: 700; cursor: pointer;" disabled>Play</button>
    <button id="recordBtn" style="padding: 10px 14px; border: 0; border-radius: 10px; background: #059669; color: white; font-weight: 700; cursor: pointer;" disabled>Start Recording</button>
    <button id="stopBtn" style="padding: 10px 14px; border: 0; border-radius: 10px; background: #dc2626; color: white; font-weight: 700; cursor: pointer;" disabled>Stop Recording</button>
    <a id="downloadLink" style="display:none; padding: 10px 14px; border-radius: 10px; background: #f59e0b; color: #111; font-weight: 700; text-decoration: none;">Download WebM</a>
  </div>
  <div id="status" style="margin-bottom: 12px;">Ready. Click Load Demo.</div>
  <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(320px, 1fr)); gap: 16px;">
    <div>
      <div style="margin-bottom: 6px; font-weight: 700;">Driving video</div>
      <video id="video" playsinline controls muted style="width: 100%; max-width: 640px; border-radius: 12px; background: black;"></video>
    </div>
    <div>
      <div style="margin-bottom: 6px; font-weight: 700;">Avatar render</div>
      <div id="canvasWrap" style="position: relative; width: 100%; max-width: 640px; aspect-ratio: 16 / 9; border-radius: 12px; overflow: hidden; background: black;"></div>
    </div>
  </div>
</div>

<script type="importmap">
{
  "imports": {
    "three": "https://cdn.skypack.dev/three@0.150.1",
    "OrbitControls": "https://cdn.skypack.dev/three@0.150.1/examples/jsm/controls/OrbitControls.js",
    "GLTFLoader": "https://cdn.skypack.dev/three@0.150.1/examples/jsm/loaders/GLTFLoader.js"
  }
}
</script>

<script type="module">
import * as THREE from "three";
import { OrbitControls } from "OrbitControls";
import { GLTFLoader } from "GLTFLoader";
import vision from "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision/vision_bundle.js";

const { FilesetResolver, FaceLandmarker } = vision;

const statusEl = document.getElementById("status");
const videoEl = document.getElementById("video");
const canvasWrap = document.getElementById("canvasWrap");
const loadBtn = document.getElementById("loadBtn");
const playBtn = document.getElementById("playBtn");
const recordBtn = document.getElementById("recordBtn");
const stopBtn = document.getElementById("stopBtn");
const downloadLink = document.getElementById("downloadLink");

const drivingVideoUrl = "__VIDEO_DATA_URL__";
const avatarUrl = "https://storage.googleapis.com/mediapipe-tasks/face_landmarker/raccoon_head.glb";
const modelUrl = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task";

let faceLandmarker = null;
let avatar = null;
let mediaRecorder = null;
let recordedChunks = [];

function setStatus(msg) {
  statusEl.textContent = msg;
}

function getViewportSizeAtDepth(camera, depth) {
  const viewportHeightAtDepth = 2 * depth * Math.tan(THREE.MathUtils.degToRad(0.5 * camera.fov));
  const viewportWidthAtDepth = viewportHeightAtDepth * camera.aspect;
  return new THREE.Vector2(viewportWidthAtDepth, viewportHeightAtDepth);
}

function createCameraPlaneMesh(camera, depth, material) {
  const viewportSize = getViewportSizeAtDepth(camera, depth);
  const cameraPlaneGeometry = new THREE.PlaneGeometry(viewportSize.width, viewportSize.height);
  cameraPlaneGeometry.translate(0, 0, -depth);
  return new THREE.Mesh(cameraPlaneGeometry, material);
}

class BasicScene {
  constructor(video) {
    this.video = video;
    this.width = 960;
    this.height = 540;
    this.scene = new THREE.Scene();
    this.camera = new THREE.PerspectiveCamera(60, this.width / this.height, 0.01, 5000);
    this.renderer = new THREE.WebGLRenderer({ antialias: true, alpha: true, preserveDrawingBuffer: true });
    this.renderer.setSize(this.width, this.height);
    this.renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
    this.renderer.outputEncoding = THREE.sRGBEncoding;
    canvasWrap.innerHTML = "";
    canvasWrap.appendChild(this.renderer.domElement);

    const ambientLight = new THREE.AmbientLight(0xffffff, 0.7);
    const directionalLight = new THREE.DirectionalLight(0xffffff, 1.0);
    directionalLight.position.set(0, 1, 1);
    this.scene.add(ambientLight);
    this.scene.add(directionalLight);

    this.camera.position.z = 0;
    this.controls = new OrbitControls(this.camera, this.renderer.domElement);
    const orbitTarget = this.camera.position.clone();
    orbitTarget.z -= 5;
    this.controls.target = orbitTarget;
    this.controls.update();

    const inputFrameTexture = new THREE.VideoTexture(video);
    inputFrameTexture.encoding = THREE.sRGBEncoding;
    const inputFramesPlane = createCameraPlaneMesh(
      this.camera,
      500,
      new THREE.MeshBasicMaterial({ map: inputFrameTexture })
    );
    this.scene.add(inputFramesPlane);

    this.callbacks = [];
    this.lastTime = 0;
    window.addEventListener("resize", () => this.resize());
    this.resize();
    this.render();
  }

  resize() {
    const wrapWidth = canvasWrap.clientWidth || 960;
    this.width = wrapWidth;
    this.height = Math.round((wrapWidth * 9) / 16);
    this.camera.aspect = this.width / this.height;
    this.camera.updateProjectionMatrix();
    this.renderer.setSize(this.width, this.height);
    this.renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
    this.renderer.render(this.scene, this.camera);
  }

  render(time = this.lastTime) {
    const delta = (time - this.lastTime) / 1000;
    this.lastTime = time;
    for (const callback of this.callbacks) {
      callback(delta);
    }
    this.renderer.render(this.scene, this.camera);
    requestAnimationFrame((t) => this.render(t));
  }
}

class Avatar {
  constructor(url, scene) {
    this.url = url;
    this.scene = scene;
    this.loader = new GLTFLoader();
    this.gltf = null;
    this.root = null;
    this.morphTargetMeshes = [];
  }

  async load() {
    return new Promise((resolve, reject) => {
      this.loader.load(
        this.url,
        (gltf) => {
          this.gltf = gltf;
          this.scene.add(gltf.scene);
          this.init(gltf);
          resolve(gltf);
        },
        undefined,
        reject
      );
    });
  }

  init(gltf) {
    gltf.scene.traverse((object) => {
      if (object.isBone && !this.root) {
        this.root = object;
      }
      if (!object.isMesh) {
        return;
      }
      object.frustumCulled = false;
      if (!object.morphTargetDictionary || !object.morphTargetInfluences) {
        return;
      }
      this.morphTargetMeshes.push(object);
    });
  }

  updateBlendshapes(blendshapesMap) {
    for (const mesh of this.morphTargetMeshes) {
      if (!mesh.morphTargetDictionary || !mesh.morphTargetInfluences) {
        continue;
      }
      for (const [name, value] of blendshapesMap.entries()) {
        if (!(name in mesh.morphTargetDictionary)) {
          continue;
        }
        const idx = mesh.morphTargetDictionary[name];
        mesh.morphTargetInfluences[idx] = value;
      }
    }
  }

  applyMatrix(matrix, scale = 40) {
    if (!this.gltf) {
      return;
    }
    matrix = matrix.clone();
    matrix.scale(new THREE.Vector3(scale, scale, scale));
    this.gltf.scene.matrixAutoUpdate = false;
    this.gltf.scene.matrix.copy(matrix);
  }
}

function retarget(blendshapes) {
  const categories = blendshapes[0].categories;
  const coefsMap = new Map();
  for (const category of categories) {
    let score = category.score;
    switch (category.categoryName) {
      case "browOuterUpLeft":
      case "browOuterUpRight":
      case "eyeBlinkLeft":
      case "eyeBlinkRight":
        score *= 1.2;
        break;
      default:
    }
    coefsMap.set(category.categoryName, score);
  }
  return coefsMap;
}

async function loadDemo() {
  setStatus("Loading MediaPipe and raccoon avatar...");
  videoEl.src = drivingVideoUrl;
  videoEl.load();

  const scene = new BasicScene(videoEl);
  avatar = new Avatar(avatarUrl, scene.scene);
  await avatar.load();

  const wasmFileset = await FilesetResolver.forVisionTasks(
    "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@latest/wasm"
  );

  faceLandmarker = await FaceLandmarker.createFromOptions(wasmFileset, {
    baseOptions: {
      modelAssetPath: modelUrl,
      delegate: "GPU"
    },
    runningMode: "VIDEO",
    outputFaceBlendshapes: true,
    outputFacialTransformationMatrixes: true,
    numFaces: 1
  });

  const onVideoFrame = async (now, metadata) => {
    if (!faceLandmarker || videoEl.paused || videoEl.ended) {
      if (!videoEl.ended) {
        videoEl.requestVideoFrameCallback(onVideoFrame);
      }
      return;
    }
    const results = faceLandmarker.detectForVideo(videoEl, now);
    const matrices = results.facialTransformationMatrixes;
    if (matrices && matrices.length > 0) {
      const matrix = new THREE.Matrix4().fromArray(matrices[0].data);
      avatar.applyMatrix(matrix, 40);
    }
    const blendshapes = results.faceBlendshapes;
    if (blendshapes && blendshapes.length > 0) {
      avatar.updateBlendshapes(retarget(blendshapes));
    }
    videoEl.requestVideoFrameCallback(onVideoFrame);
  };

  videoEl.addEventListener("play", () => {
    videoEl.requestVideoFrameCallback(onVideoFrame);
  });

  videoEl.addEventListener("ended", () => {
    if (mediaRecorder && mediaRecorder.state !== "inactive") {
      mediaRecorder.stop();
    }
  });

  playBtn.disabled = false;
  recordBtn.disabled = false;
  setStatus("Loaded. Click Play.");

  const stream = scene.renderer.domElement.captureStream(30);
  mediaRecorder = new MediaRecorder(stream, { mimeType: "video/webm" });
  recordedChunks = [];
  mediaRecorder.ondataavailable = (event) => {
    if (event.data && event.data.size > 0) {
      recordedChunks.push(event.data);
    }
  };
  mediaRecorder.onstop = () => {
    const blob = new Blob(recordedChunks, { type: "video/webm" });
    const url = URL.createObjectURL(blob);
    downloadLink.href = url;
    downloadLink.download = "raccoon_avatar_control.webm";
    downloadLink.style.display = "inline-block";
    setStatus("Recording finished. Download link is ready.");
  };
}

loadBtn.onclick = async () => {
  loadBtn.disabled = true;
  try {
    await loadDemo();
  } catch (err) {
    console.error(err);
    setStatus("Load failed: " + err);
    loadBtn.disabled = false;
  }
};

playBtn.onclick = async () => {
  downloadLink.style.display = "none";
  await videoEl.play();
  setStatus("Playing.");
};

recordBtn.onclick = async () => {
  if (!mediaRecorder) {
    return;
  }
  recordedChunks = [];
  downloadLink.style.display = "none";
  videoEl.currentTime = 0;
  await videoEl.play();
  mediaRecorder.start();
  recordBtn.disabled = true;
  stopBtn.disabled = false;
  setStatus("Recording started.");
};

stopBtn.onclick = () => {
  if (mediaRecorder && mediaRecorder.state !== "inactive") {
    mediaRecorder.stop();
  }
  recordBtn.disabled = false;
  stopBtn.disabled = true;
  setStatus("Stopping recording...");
};
</script>
'''

html = html.replace("__VIDEO_DATA_URL__", video_data_url)
HTML(html)
